# V96 - Endpoints concat

Pre-registered plan: first-only and last-only each lose alone, but their union may drop the noisy middle while keeping both intent endpoints. First plus last concatenated faces all-chat under identical folds with a fresh seed. Same locks. If the middle messages are dilution, this wins; if they accumulate signal, it loses like the extremes.

In [ ]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedGroupKFold

SEED = 2121
N_SPLITS = 5
GATE_FLOOR = 0.002
rng = np.random.RandomState(SEED)

CANDIDATES = [Path("/kaggle/input/intelligo-data/data"), Path("data")]
DATA = next((p for p in CANDIDATES if (p / "train_relevance.csv").exists()), None)
assert DATA is not None, "train_relevance.csv not found"
OUT = Path("/kaggle/working" if Path("/kaggle/working").exists() else "results/v96")
OUT.mkdir(parents=True, exist_ok=True)
print("DATA:", DATA.resolve())

## Endpoints vs all-chat, same folds

One shared vocabulary so dimensions match exactly. Single-chat users are identical across views by construction.

In [ ]:
MODS = [f"M_{i:03d}" for i in range(1, 18)]
train = pd.read_csv(DATA / "train_relevance.csv")
test_ids = pd.read_csv(DATA / "test.csv")["user_id"].tolist()
chats = pd.read_csv(DATA / "chat_history.csv", parse_dates=["timestamp"])
assert not (set(train["user_id"]) & set(test_ids))
Y = train[MODS].to_numpy(float)
n_train = len(train)
users = list(train["user_id"])
chats = chats.sort_values(["user_id", "timestamp"])
by_user = chats.groupby("user_id")["user_chat_text"].apply(list)
first = [str(by_user.get(u, [""])[0]) for u in users]
last = [str(by_user.get(u, [""])[-1]) for u in users]
full = [" ".join(str(t) for t in by_user.get(u, [])) for u in users]
ends = [f + " " + l for f, l in zip(first, last)]
VF = TfidfVectorizer(max_features=5000, ngram_range=(1, 1), min_df=3).fit(first + last + full)
FE, FF = VF.transform(ends).toarray(), VF.transform(full).toarray()

def template_group(texts):
    if not texts:
        return "NO_CHAT"
    return re.sub(r"\d+", "#", " ".join(str(texts[0]).lower().split()[:6]))
fold_groups = np.array([template_group(by_user.get(u, [])) for u in users])
strata = Y.argmax(1)

def ndcg5(yt, ys):
    order = np.argsort(-ys, axis=1)
    rel = np.take_along_axis(yt, order, 1)[:, :5]
    disc = 1.0 / np.log2(np.arange(2, 7))
    dcg = (rel * disc).sum(1)
    ideal = np.sort(yt, axis=1)[:, ::-1][:, :5]
    idcg = (ideal * disc).sum(1)
    return np.where(idcg > 0, dcg / np.maximum(idcg, 1e-12), 0.0)

def ridge_oof(Xtr, alpha=1.0):
    oof = np.zeros_like(Y)
    fids = np.zeros(n_train, int)
    skf = StratifiedGroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    for fi, (tr, va) in enumerate(skf.split(Xtr, strata, fold_groups)):
        for j in range(len(MODS)):
            m = Ridge(alpha=alpha)
            m.fit(Xtr[tr], Y[tr, j])
            oof[va, j] = m.predict(Xtr[va])
        fids[va] = fi + 1
    return oof, fids

oof_f, fid = ridge_oof(FF)
oof_e, _ = ridge_oof(FE)
base = ndcg5(Y, oof_f)
d = ndcg5(Y, oof_e) - base
lo, hi = np.quantile([rng.choice(d, len(d), replace=True).mean() for _ in range(2000)], [0.025, 0.975])
wf = min(ndcg5(Y[fid == k], oof_e[fid == k]).mean() - base[fid == k].mean() for k in range(1, N_SPLITS + 1))
print(f"all={base.mean():.5f} endpoints={ndcg5(Y, oof_e).mean():.5f} delta={d.mean():+.5f} CI95=[{lo:+.5f},{hi:+.5f}] worst-fold={wf:+.5f}")
print("ADOPT endpoints" if (lo > GATE_FLOOR and wf >= 0) else "KEEP all-chat; middle messages accumulate signal")